# Crawler: WebSocket to Disk (Direct)

This notebook crawls stock data from SSI WebSocket and saves directly to disk, without Redis.

## Imports

In [16]:
import os
import json
import asyncio
import websockets
import pandas as pd
from datetime import datetime
from IPython.display import clear_output

## Configuration

In [17]:
# Config
DATA_DIR = "./data/direct"  # Direct crawl path
BATCH_SIZE = 100
WS_URL = "wss://iboard-pushstream.ssi.com.vn/realtime"

STOCKS = ["41I1G3000", "41I1G4000"]  # Replace with actual symbols

## Writer Logic

In [18]:
import pyarrow.feather as feather
import pandas as pd

def save_batch(buffer):
    """Save buffer to Feather files, grouped by stock."""
    if not buffer:
        return
    
    # Group by stock
    by_stock = {}
    for row in buffer:
        stk = row.get('stk')
        if stk not in by_stock:
            by_stock[stk] = []
        by_stock[stk].append(row)

    # Save each stock to its own file
    for stk, rows in by_stock.items():
        # Create directory for stock
        dir_path = os.path.join(DATA_DIR, stk)
        os.makedirs(dir_path, exist_ok=True)

        # File path: data/{stk}/{date}.fea
        file_path = os.path.join(dir_path, f"{date_str}.fea")
        
        df_new = pd.DataFrame(rows)
        
        # Append mode: read existing, concat, write
        if os.path.exists(file_path):
            df_existing = pd.read_feather(file_path)
            df = pd.concat([df_existing, df_new], ignore_index=True)
        else:
            df = df_new
        
        # Save to feather
        feather.write_feather(df, file_path)
        
        print(f"Saved {len(rows)} rows for {stk} to {file_path}")

def read_message(msg):
    """Parse pipe-delimited message from SSI."""
    def sp(s, mode=0):
        try:
            return float(s) if mode == 0 else int(s)
        except:
            return 0

    L = msg.split('|')
    if len(L) >= 102:
        tm = datetime.now()
        stk = str(L[1][2:])
        
        # Check if derivative (phái sinh)
        is_derivative = stk in ("41I1FA000", "41I1FB000")
        
        # Different indices for derivatives vs stocks
        if is_derivative:
            # Derivatives
            idx_ttqt = 49  # tong_kl
            idx_high = 44
            idx_low = 42
        else:
            # Stocks
            idx_ttqt = 54  # tong_kl
            idx_high = 44
            idx_low = 46
        
        trdg, diff,chng = sp(L[42]) / 1000, sp(L[52]) / 1000, sp(L[53])
        high, low = sp(L[idx_high]) / 1000, sp(L[idx_low]) / 1000
        nstr, ttqt = sp(L[43], 1), sp(L[idx_ttqt], 1)
        ceil, floor, adj = sp(L[59]) / 1000, sp(L[60]) / 1000, sp(L[61]) / 1000
        b1, b2, b3 = sp(L[2]) / 1000, sp(L[4]) / 1000, sp(L[6]) / 10000
        s1, s2, s3 = sp(L[22]) / 1000, sp(L[24]) / 1000, sp(L[26]) / 10000
        nsb1, nsb2, nsb3 = sp(L[3], 1), sp(L[5], 1), sp(L[7], 1)
        nss1, nss2, nss3 = sp(L[23], 1), sp(L[25], 1), sp(L[27], 1)
        fb, fs, fr = sp(L[48], 1), sp(L[50], 1), sp(L[65], 1)

        return {
            'tick': tm.isoformat(),
            'stk': stk,
            'ceil': ceil,
            'floor': floor,
            'adj': adj,
            'b3': b3, 'nsb3': nsb3,
            'b2': b2, 'nsb2': nsb2,
            'b1': b1, 'nsb1': nsb1,
            'trdg': trdg, 'nstr': nstr, 'diff': diff, 'chng':chng,
            's1': s1, 'nss1': nss1,
            's2': s2, 'nss2': nss2,
            's3': s3, 'nss3': nss3,
            'ttqt': ttqt,
            'high': high, 'low': low,
            'fb': fb, 'fs': fs, 'fr': fr
        }
    return None

## Crawler Logic

In [19]:
async def run_crawler():
    os.makedirs(DATA_DIR, exist_ok=True)

    global date_str
    date_str = datetime.now().strftime("%Y-%m-%d")
    buffer = []

    print(f"Connecting to {WS_URL}...")

    async with websockets.connect(WS_URL) as ws:
        print("Connected!")

        subscribe_msg = {
            "type": "sub",
            "topic": "stockRealtimeByListV2",
            "component": "priceTableEquities",
            "variables": STOCKS
        }
        await ws.send(json.dumps(subscribe_msg))
        print(f"Subscribed to {STOCKS}")

        try:
            while True:
                message = await ws.recv()
                data = read_message(message)

                if data:
                    buffer.append(data)
                    print(f"Received: {data['stk']} @ {data['tick']}", end='\r')

                if len(buffer) >= BATCH_SIZE:
                    save_batch(buffer)
                    buffer = []  # Clear buffer

        except Exception as e:
            print(f"Error: {e}")
            # Save remaining buffer on exit
            if buffer:
                save_batch(buffer)

In [20]:
# Run the crawler
await run_crawler()

Connecting to wss://iboard-pushstream.ssi.com.vn/realtime...
Connected!
Subscribed to ['41I1G3000', '41I1G4000']
Saved 78 rows for 41I1G3000 to ./data/direct/41I1G3000/2026-03-12.fea
Saved 22 rows for 41I1G4000 to ./data/direct/41I1G4000/2026-03-12.fea
Saved 90 rows for 41I1G3000 to ./data/direct/41I1G3000/2026-03-12.fea
Saved 10 rows for 41I1G4000 to ./data/direct/41I1G4000/2026-03-12.fea
Saved 87 rows for 41I1G3000 to ./data/direct/41I1G3000/2026-03-12.fea
Saved 13 rows for 41I1G4000 to ./data/direct/41I1G4000/2026-03-12.fea
Saved 97 rows for 41I1G3000 to ./data/direct/41I1G3000/2026-03-12.fea
Saved 3 rows for 41I1G4000 to ./data/direct/41I1G4000/2026-03-12.fea
Saved 87 rows for 41I1G3000 to ./data/direct/41I1G3000/2026-03-12.fea
Saved 13 rows for 41I1G4000 to ./data/direct/41I1G4000/2026-03-12.fea
Saved 18 rows for 41I1G4000 to ./data/direct/41I1G4000/2026-03-12.fea
Saved 82 rows for 41I1G3000 to ./data/direct/41I1G3000/2026-03-12.fea
Saved 84 rows for 41I1G3000 to ./data/direct/41I